# Grid Search Analysis

Visualize results from the (σ₀, σ, η) parameter sweep.
Loads pre-computed per-triple `.npz` files and the gathered CSV/grid arrays.

Plots:
1. d' heatmaps (two slice orientations)
2. ISI decay curves for triples with strongest memory decay
3. ROC curves per ISI for selected triples
4. Hit vs FA score distributions
5. Parameter sensitivity (marginal d' curves)
6. Temporal dynamics (hit score vs trial position)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Point this to your grid search output directory ──
RESULTS_DIR = '../reports/figures/2d_grid_search'

# Load grid arrays
grid = np.load(os.path.join(RESULTS_DIR, 'grid_search_results.npz'))
sigma0_grid = grid['sigma0_grid']
sigma_grid = grid['sigma_grid']
eta_grid = grid['eta_grid']
ISI_VALUES = tuple(grid['isi_values'].astype(int))

results = {isi: grid[f'dprime_isi{isi}'] for isi in ISI_VALUES}

# Load master CSV
df = pd.read_csv(os.path.join(RESULTS_DIR, 'grid_search_master.csv'))

PER_TRIPLE_DIR = os.path.join(RESULTS_DIR, 'per_triple')

def load_triple(s0, sig, eta):
    fname = f's0={s0:.3f}_sig={sig:.3f}_eta={eta:.3f}.npz'
    return np.load(os.path.join(PER_TRIPLE_DIR, fname), allow_pickle=True)

print(f'Grid: {len(sigma0_grid)} x {len(sigma_grid)} x {len(eta_grid)} = {len(df)} triples')
print(f'ISI values: {ISI_VALUES}')
print(f'sigma0: {sigma0_grid}')
print(f'sigma:  {sigma_grid}')
print(f'eta:    {eta_grid}')

## d' Heatmaps: d'(σ, η) sliced by σ₀

Rows = ISI, Columns = σ₀ level. Each panel shows d' as a function of diffusive noise (σ) and drift step (η).

In [ ]:
s0_slice_indices = [0, 2, 4, 6]  # 0.0, 0.25, 0.75, 1.5

fig, axes = plt.subplots(len(ISI_VALUES), len(s0_slice_indices),
                         figsize=(4 * len(s0_slice_indices), 3.5 * len(ISI_VALUES)),
                         sharex=True, sharey=True)

vmin = 0
vmax = min(np.nanmax([results[isi] for isi in ISI_VALUES]), 5.0)

for row, isi in enumerate(ISI_VALUES):
    for col, i_s0 in enumerate(s0_slice_indices):
        ax = axes[row, col]
        data = results[isi][i_s0, :, :]  # [n_sigma, n_eta]
        im = ax.imshow(data, aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax, cmap='viridis',
                       extent=[eta_grid[0], eta_grid[-1],
                               sigma_grid[0], sigma_grid[-1]])
        if row == 0:
            ax.set_title(f'\u03c3\u2080={sigma0_grid[i_s0]:.2f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={isi}\n\u03c3 (diffusive)', fontsize=10)
        if row == len(ISI_VALUES) - 1:
            ax.set_xlabel('\u03b7 (drift step)', fontsize=10)

fig.suptitle("d' across parameter space (rows=ISI, cols=\u03c3\u2080)", fontsize=13, y=1.01)
fig.colorbar(im, ax=axes, shrink=0.6, label="d'")
fig.tight_layout()
plt.show()

## d' Heatmaps: d'(σ₀, η) sliced by σ

Complementary view: fix diffusive noise, sweep encoding noise vs drift.

In [ ]:
sig_slice_indices = [0, 2, 4, 6]  # 0.0, 0.05, 0.15, 0.3

fig, axes = plt.subplots(len(ISI_VALUES), len(sig_slice_indices),
                         figsize=(4 * len(sig_slice_indices), 3.5 * len(ISI_VALUES)),
                         sharex=True, sharey=True)

for row, isi in enumerate(ISI_VALUES):
    for col, i_sig in enumerate(sig_slice_indices):
        ax = axes[row, col]
        data = results[isi][:, i_sig, :]  # [n_sigma0, n_eta]
        im = ax.imshow(data, aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax, cmap='viridis',
                       extent=[eta_grid[0], eta_grid[-1],
                               sigma0_grid[0], sigma0_grid[-1]])
        if row == 0:
            ax.set_title(f'\u03c3={sigma_grid[i_sig]:.3f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={isi}\n\u03c3\u2080 (encoding)', fontsize=10)
        if row == len(ISI_VALUES) - 1:
            ax.set_xlabel('\u03b7 (drift step)', fontsize=10)

fig.suptitle("d' across parameter space (rows=ISI, cols=\u03c3)", fontsize=13, y=1.01)
fig.colorbar(im, ax=axes, shrink=0.6, label="d'")
fig.tight_layout()
plt.show()

## ISI Decay Curves: Triples with Strongest Memory Decay

Select triples that are (a) above chance at ISI=0 and (b) show the largest d' drop from ISI=0 to ISI=16.
Triples that are flat or at chance everywhere are excluded.

In [ ]:
# Compute ISI decay and filter
df['delta_dp'] = df['dprime_isi0'] - df['dprime_isi16']

interesting = df[
    (df['dprime_isi0'] > 0.5) &        # above chance at ISI=0
    (df['delta_dp'].abs() > 0.1)        # non-flat ISI curve
].copy()

top = interesting.nlargest(6, 'delta_dp')
print(f'{len(interesting)} interesting triples (of {len(df)} total)')
print(f'Top {len(top)} by ISI decay:')
print(top[['sigma0', 'sigma', 'eta', 'dprime_isi0', 'dprime_isi2', 'dprime_isi16', 'delta_dp']].to_string(index=False))

# Plot
isi_arr = np.array(ISI_VALUES)
colors = plt.cm.tab10(np.linspace(0, 1, len(top)))

fig, ax = plt.subplots(figsize=(7, 5))
for i, (_, row) in enumerate(top.iterrows()):
    data = load_triple(row['sigma0'], row['sigma'], row['eta'])
    dps = [float(data[f'dprime_isi{isi}']) for isi in ISI_VALUES]
    sems = [float(data[f'dprime_sem_isi{isi}']) for isi in ISI_VALUES]
    label = f'\u03c3\u2080={row["sigma0"]:.2f}, \u03c3={row["sigma"]:.3f}, \u03b7={row["eta"]:.3f}'
    ax.errorbar(isi_arr, dps, yerr=sems, marker='o', capsize=4,
                color=colors[i], label=label, linewidth=2)

ax.set_xlabel('ISI', fontsize=12)
ax.set_ylabel("d'", fontsize=12)
ax.set_title('ISI Decay: Strongest Memory Degradation', fontsize=13)
ax.legend(fontsize=8, loc='best')
ax.set_xticks(isi_arr)
fig.tight_layout()
plt.show()

## ROC Curves for Selected Triples

Full ROC (FPR vs TPR) per ISI for the top decay triples. Shows whether the ISI effect is driven by hits shifting vs FA shifting.

In [ ]:
isi_colors = {0: 'green', 2: 'blue', 16: 'orange'}
n_triples = min(len(top), 6)
ncols = 3
nrows = int(np.ceil(n_triples / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes = np.atleast_2d(axes)

for idx, (_, row) in enumerate(top.head(n_triples).iterrows()):
    ax = axes.flat[idx]
    data = load_triple(row['sigma0'], row['sigma'], row['eta'])
    for isi in ISI_VALUES:
        fpr = data[f'roc_fpr_isi{isi}']
        tpr = data[f'roc_tpr_isi{isi}']
        auc = float(data[f'auc_isi{isi}'])
        if len(fpr) > 0:
            ax.plot(fpr, tpr, color=isi_colors[isi], linewidth=2,
                    label=f'ISI={isi} (AUC={auc:.2f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.5, alpha=0.5)
    ax.set_title(f'\u03c3\u2080={row["sigma0"]:.2f}, \u03c3={row["sigma"]:.3f}, \u03b7={row["eta"]:.3f}',
                 fontsize=10)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=8)
    ax.set_aspect('equal')

# Hide unused axes
for idx in range(n_triples, nrows * ncols):
    axes.flat[idx].set_visible(False)

fig.suptitle('ROC Curves per ISI', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## Score Distributions: Hit vs FA

Overlapping histograms showing how hit score distributions shift relative to FA scores across ISI conditions.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_2d(axes)

for idx, (_, row) in enumerate(top.head(n_triples).iterrows()):
    ax = axes.flat[idx]
    data = load_triple(row['sigma0'], row['sigma'], row['eta'])
    fa = data['fa_scores']

    # Determine bin range across all scores
    all_scores = [fa]
    for isi in ISI_VALUES:
        h = data[f'hit_scores_isi{isi}']
        if len(h) > 0:
            all_scores.append(h)
    combined = np.concatenate(all_scores)
    bins = np.linspace(np.percentile(combined, 1), np.percentile(combined, 99), 40)

    ax.hist(fa, bins=bins, alpha=0.4, color='gray', label='FA', density=True)
    for isi, color in isi_colors.items():
        h = data[f'hit_scores_isi{isi}']
        if len(h) > 0:
            ax.hist(h, bins=bins, alpha=0.4, color=color,
                    label=f'Hit ISI={isi}', density=True)

    ax.set_title(f'\u03c3\u2080={row["sigma0"]:.2f}, \u03c3={row["sigma"]:.3f}, \u03b7={row["eta"]:.3f}',
                 fontsize=10)
    ax.set_xlabel('Cosine distance')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

for idx in range(n_triples, nrows * ncols):
    axes.flat[idx].set_visible(False)

fig.suptitle('Hit vs FA Score Distributions', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## Parameter Sensitivity: Marginal d' Curves

Average d' over two of three parameters to see which parameter matters most for each ISI.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# d' vs sigma0 (avg over sigma, eta)
ax = axes[0]
for isi in ISI_VALUES:
    marginal = np.nanmean(results[isi], axis=(1, 2))  # avg over sigma, eta
    ax.plot(sigma0_grid, marginal, 'o-', linewidth=2, label=f'ISI={isi}')
ax.set_xlabel('\u03c3\u2080 (encoding noise)', fontsize=11)
ax.set_ylabel("Mean d'", fontsize=11)
ax.set_title('Marginal over \u03c3, \u03b7', fontsize=12)
ax.legend()

# d' vs sigma (avg over sigma0, eta)
ax = axes[1]
for isi in ISI_VALUES:
    marginal = np.nanmean(results[isi], axis=(0, 2))  # avg over sigma0, eta
    ax.plot(sigma_grid, marginal, 'o-', linewidth=2, label=f'ISI={isi}')
ax.set_xlabel('\u03c3 (diffusive noise)', fontsize=11)
ax.set_ylabel("Mean d'", fontsize=11)
ax.set_title('Marginal over \u03c3\u2080, \u03b7', fontsize=12)
ax.legend()

# d' vs eta (avg over sigma0, sigma)
ax = axes[2]
for isi in ISI_VALUES:
    marginal = np.nanmean(results[isi], axis=(0, 1))  # avg over sigma0, sigma
    ax.plot(eta_grid, marginal, 'o-', linewidth=2, label=f'ISI={isi}')
ax.set_xlabel('\u03b7 (drift step)', fontsize=11)
ax.set_ylabel("Mean d'", fontsize=11)
ax.set_title('Marginal over \u03c3\u2080, \u03c3', fontsize=12)
ax.legend()

fig.suptitle('Parameter Sensitivity: Marginal d\' Curves', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Temporal Dynamics: Hit Score vs Trial Position

Does memory degrade over the course of a sequence? Scatter hit scores against their trial index, with a rolling-mean trend line.

In [ ]:
# Use the top-1 decay triple
row = top.iloc[0]
data = load_triple(row['sigma0'], row['sigma'], row['eta'])

fig, ax = plt.subplots(figsize=(8, 5))

for isi, color in isi_colors.items():
    scores = data[f'hit_scores_isi{isi}']
    times = data[f'hit_timestamps_isi{isi}']
    if len(scores) == 0:
        continue
    ax.scatter(times, scores, alpha=0.3, s=15, color=color, label=f'ISI={isi}')

    # Rolling mean trend line
    if len(scores) > 5:
        order = np.argsort(times)
        t_sorted = times[order].astype(float)
        s_sorted = scores[order]
        window = max(len(s_sorted) // 10, 5)
        kernel = np.ones(window) / window
        smoothed = np.convolve(s_sorted, kernel, mode='valid')
        t_smooth = np.convolve(t_sorted, kernel, mode='valid')
        ax.plot(t_smooth, smoothed, color=color, linewidth=2, alpha=0.8)

ax.set_xlabel('Trial position', fontsize=12)
ax.set_ylabel('Hit score (cosine distance)', fontsize=12)
ax.set_title(
    f'Temporal dynamics: \u03c3\u2080={row["sigma0"]:.2f}, '
    f'\u03c3={row["sigma"]:.3f}, \u03b7={row["eta"]:.3f}',
    fontsize=13)
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()